In [33]:
import pandas as pd
import geopandas as gpd
import json
import osmnx as ox
import pydeck as pdk

In [34]:
# Load the data
with open('data/shelters.json', 'r', encoding='utf-8') as file:
    data = json.load(file)


In [35]:
df = pd.DataFrame(data['elements'])
pdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']))
pdf.head()

,type,id,lat,lon,tags,geometry
0,node,294222713,50.817455,14.705947,"{'amenity': 'shelter', 'ele': '480', 'name': '...",POINT (14.70595 50.81745)
1,node,326124507,49.179924,13.211484,"{'amenity': 'shelter', 'shelter_type': 'picnic...",POINT (13.21148 49.17992)
2,node,371161040,50.021147,14.285731,"{'amenity': 'shelter', 'shelter_type': 'picnic...",POINT (14.28573 50.02115)
3,node,373413231,49.987023,14.283100,"{'amenity': 'shelter', 'shelter_type': 'picnic...",POINT (14.2831 49.98702)
4,node,373462135,49.356274,16.599385,"{'amenity': 'shelter', 'shelter_type': 'picnic...",POINT (16.59939 49.35627)


In [36]:
# Define the center of the map
center = (49.8022514, 15.6252330)

In [37]:
# Define the icon
icon = {
    "url": "https://cdn-icons-png.flaticon.com/128/2776/2776067.png",
    "width": 128,
    "height": 128
}

# Add icon to each record
pdf['icon_settings']= None
for i in pdf.index:
    pdf.at[i, 'icon_settings'] = icon

# Define the icon layer
icon_layer = pdk.Layer(
    type='IconLayer',
    data=pdf,
    get_icon='icon_settings',
    get_size=4,
    pickable=True,
    size_scale=5,
    get_position=['lon', 'lat']
)

# Set the viewport location and zoom
view_state = pdk.ViewState(
    longitude=center[1],
    latitude=center[0],
    zoom=6,
    min_zoom=5,
    max_zoom=15
)

# Define text and appearance of tooltips
tooltip = {
   "html": "<b>Latitude:</b> {lat} <br/> <b>Longitude:</b> {lon}",
   "style": {
        "backgroundColor": "steelblue",
        "color": "white"
   }
}

# Create the map
map = pdk.Deck(layers=[icon_layer], initial_view_state=view_state, map_style='road', tooltip=tooltip)

# Save the map into html file
map.to_html('map.html')
